# Automated cluster exploration notebook

This notebook starts from the RAPIDS clustered AnnData output and produces cluster-exploration tables and plots only.

Scientific logic preserved from the original notebook:
- no new clustering
- no extra filtering
- dotplots use the same scaled representation used for RAPIDS clustering
- matrixplots use `arcsinh_full` intensities
- only selected Leiden resolutions and UMAP `min_dist` values are plotted
- nothing is added to the final AnnData

In [ ]:
# ==============================================================================
# PARAMETERS — overwritten by Papermill / Nextflow
# ==============================================================================

SAMPLE_INDEX = 1
SAMPLE_ID = "sample"
SAMPLE_DIR_NAME = "001_sample"

INPUT_PATH = "rapids_clustered.h5ad"
OUTPUT_DIR = "cluster_exploration_output"

# Select only the cluster/UMAP combinations you want to visualize.
# Accepts either strings ("0_5", "0_5,1_0") or Python lists.
SELECTED_NORMALIZATIONS = "arcsinh,log1p"
SELECTED_RESOLUTIONS = "0_5"
SELECTED_UMAP_MIN_DISTS = "0_3"

# Marker UMAP style
MARKER_UMAP_CMAP = "RdYlBu_r"
MARKER_UMAP_NCOLS = 3
MARKER_UMAP_POINT_SIZE = 1.5

# Cluster UMAP style
CLUSTER_UMAP_POINT_SIZE = 1.5

# Section 5 expression checks
DOTPLOT_CMAP = "Reds"
MATRIXPLOT_CMAP = "viridis"
STANDARD_SCALE = "var"
MATRIXPLOT_DENDROGRAM = True

# Spatial plots
SPATIAL_POINT_SIZE = 0.8
SPATIAL_ALPHA = 0.75
SPATIAL_PALETTE = "tab20"

# Tables / figures
PDF_DPI = 300
SAVE_PNG_COPY = False

In [ ]:
# ==============================================================================
# SECTION 1: IMPORTS, OUTPUT FOLDERS, LOAD RAPIDS ANNDATA
# ==============================================================================

from pathlib import Path
import math
import h5py
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from scipy import sparse

try:
    from great_tables import GT, md as gt_md
    GREAT_TABLES_AVAILABLE = True
except Exception:
    GREAT_TABLES_AVAILABLE = False

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor="white")


def normalize_token(x):
    return str(x).strip().replace(".", "_")


def pretty_token(x):
    return str(x).replace("_", ".")


def as_list(x):
    if x is None:
        return []
    if isinstance(x, (list, tuple)):
        return [normalize_token(v) for v in x]
    return [
        normalize_token(v)
        for v in str(x).replace(";", ",").split(",")
        if str(v).strip()
    ]


def save_df_as_pdf(df, path_pdf, title=None, max_rows=None, font_size=8):
    path_pdf = Path(path_pdf)
    path_pdf.parent.mkdir(parents=True, exist_ok=True)

    display_df = df.copy()
    if max_rows is not None:
        display_df = display_df.head(max_rows)

    n_rows = max(1, len(display_df))
    n_cols = max(1, len(display_df.columns))

    fig_w = max(8, min(24, n_cols * 1.7))
    fig_h = max(2.5, min(40, n_rows * 0.32 + 1.8))

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis("off")

    if title:
        ax.set_title(title, fontsize=12, fontweight="bold", loc="left", pad=12)

    table = ax.table(
        cellText=display_df.astype(str).values,
        colLabels=list(display_df.columns),
        cellLoc="left",
        colLoc="left",
        loc="center",
    )

    table.auto_set_font_size(False)
    table.set_fontsize(font_size)
    table.scale(1, 1.25)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold")
            cell.set_facecolor("#f0f0f0")

    plt.tight_layout()
    fig.savefig(path_pdf, dpi=PDF_DPI, bbox_inches="tight")
    plt.close(fig)


def display_table(df, title=None, subtitle=None):
    if GREAT_TABLES_AVAILABLE:
        gt = GT(df)
        if title is not None:
            gt = gt.tab_header(
                title=gt_md(f"**{title}**"),
                subtitle=subtitle or ""
            )
        display(gt.tab_options(table_background_color="white", heading_align="left"))
    else:
        if title:
            print(title)
        display(df)


# Normalize selected parameters
SELECTED_NORMALIZATIONS = [
    str(v).strip()
    for v in str(SELECTED_NORMALIZATIONS).replace(";", ",").split(",")
    if str(v).strip()
] if not isinstance(SELECTED_NORMALIZATIONS, (list, tuple)) else list(SELECTED_NORMALIZATIONS)

SELECTED_RESOLUTIONS = as_list(SELECTED_RESOLUTIONS)
SELECTED_UMAP_MIN_DISTS = as_list(SELECTED_UMAP_MIN_DISTS)

print("Selected normalizations:", SELECTED_NORMALIZATIONS)
print("Selected Leiden resolutions:", SELECTED_RESOLUTIONS)
print("Selected UMAP min_dist values:", SELECTED_UMAP_MIN_DISTS)


# Output folders
OUTPUT_DIR = Path(OUTPUT_DIR)
TABLE_DIR = OUTPUT_DIR / "tables"
DIST_DIR = OUTPUT_DIR / "intensity_distributions"
UMAP_MARKER_DIR = OUTPUT_DIR / "umap_marker_intensity"
UMAP_CLUSTER_DIR = OUTPUT_DIR / "umap_clusters"
QC_DIR = OUTPUT_DIR / "quality_checks"
SPATIAL_DIR = OUTPUT_DIR / "spatial"
REPORT_DIR = OUTPUT_DIR / "report"

for d in [TABLE_DIR, DIST_DIR, UMAP_MARKER_DIR, UMAP_CLUSTER_DIR, QC_DIR, SPATIAL_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)


# AnnData compatibility fix
INPUT_PATH = str(INPUT_PATH)

with h5py.File(INPUT_PATH, "a") as f:
    if "uns/log1p" in f:
        del f["uns/log1p"]
        print("Patched: removed /uns/log1p from RAPIDS AnnData")

adata = ad.read_h5ad(INPUT_PATH)

print("=" * 80)
print("Loaded AnnData")
print(adata)
print("=" * 80)

print("obs columns:", list(adata.obs.columns))
print("obsm keys:", list(adata.obsm.keys()))
print("layers:", list(adata.layers.keys()))
print("uns keys:", list(adata.uns.keys()))

if "dapi_qc" in adata.uns:
    print("\nDAPI QC metadata:")
    print(adata.uns["dapi_qc"])

lineage_markers = list(adata.uns["lineage_markers"])
leiden_cols = sorted([c for c in adata.obs.columns if c.startswith("leiden_")])
umap_keys = sorted([k for k in adata.obsm.keys() if k.startswith("X_umap_")])

print(f"\nLineage markers ({len(lineage_markers)}): {lineage_markers}")
print(f"Leiden columns ({len(leiden_cols)}): {leiden_cols}")
print(f"UMAP embeddings ({len(umap_keys)}): {umap_keys}")

In [ ]:
# ==============================================================================
# SECTION 2: AVAILABLE AND SELECTED CLUSTER CHOICES
# ==============================================================================

def leiden_key(norm, res):
    return f"leiden_{norm}_res_{res}"


def umap_key(norm, md):
    return f"X_umap_{norm}_md_{md}"


def parse_leiden_key(key):
    norm = key.split("_")[1]
    res = key.split("res_")[-1]
    return norm, res


def cluster_sort_key(x):
    try:
        return int(x)
    except Exception:
        return str(x)


def cluster_order(leiden_key_value):
    labels = adata.obs[leiden_key_value].astype(str).unique().tolist()
    return sorted(labels, key=cluster_sort_key)


def n_clusters(leiden_key_value):
    return int(adata.obs[leiden_key_value].nunique())


available_rows = []
for key in leiden_cols:
    norm, res = parse_leiden_key(key)
    available_mds = []
    for ukey in umap_keys:
        prefix = f"X_umap_{norm}_md_"
        if ukey.startswith(prefix):
            available_mds.append(ukey.replace(prefix, ""))

    available_rows.append({
        "normalization": norm,
        "resolution": pretty_token(res),
        "resolution_token": res,
        "n_clusters": n_clusters(key),
        "leiden_key": key,
        "available_umap_min_dist": ", ".join(pretty_token(x) for x in sorted(available_mds)),
    })

available_choices = pd.DataFrame(available_rows).sort_values(
    ["normalization", "resolution_token"]
)

display_table(
    available_choices,
    title="Available Leiden cluster choices",
    subtitle="Only selected choices are plotted below."
)

save_df_as_pdf(
    available_choices,
    TABLE_DIR / "available_cluster_choices.pdf",
    title=f"{SAMPLE_ID} - available Leiden cluster choices",
)

selected_rows = []

for norm in SELECTED_NORMALIZATIONS:
    for res in SELECTED_RESOLUTIONS:
        key = leiden_key(norm, res)

        if key not in adata.obs:
            print(f"Skipping selected Leiden key not found: {key}")
            continue

        for md in SELECTED_UMAP_MIN_DISTS:
            ukey = umap_key(norm, md)

            if ukey not in adata.obsm:
                print(f"Skipping selected UMAP key not found: {ukey}")
                continue

            selected_rows.append({
                "normalization": norm,
                "resolution": pretty_token(res),
                "resolution_token": res,
                "umap_min_dist": pretty_token(md),
                "umap_min_dist_token": md,
                "n_clusters": n_clusters(key),
                "leiden_key": key,
                "umap_key": ukey,
            })

selected_choices = pd.DataFrame(selected_rows)

if selected_choices.empty:
    raise ValueError(
        "No selected cluster/UMAP choices are available. "
        "Check SELECTED_NORMALIZATIONS, SELECTED_RESOLUTIONS, and SELECTED_UMAP_MIN_DISTS."
    )

display_table(
    selected_choices,
    title="Selected cluster choices for this run",
    subtitle="Only these combinations will be plotted."
)

save_df_as_pdf(
    selected_choices,
    TABLE_DIR / "selected_cluster_choices.pdf",
    title=f"{SAMPLE_ID} - selected cluster choices",
)

selected_leiden_keys = selected_choices["leiden_key"].drop_duplicates().tolist()
print("Selected Leiden keys:", selected_leiden_keys)

In [ ]:
# ==============================================================================
# SECTION 3: QC SUMMARY TABLES AND INTENSITY DISTRIBUTIONS
# ==============================================================================

def layer_to_array(layer_key, markers):
    if layer_key not in adata.layers:
        raise KeyError(f"{layer_key} not found in adata.layers. Available: {list(adata.layers.keys())}")

    X = adata[:, markers].layers[layer_key]

    if sparse.issparse(X):
        X = X.toarray()
    else:
        X = np.asarray(X)

    return X


# Raw intensity statistics
if "raw" in adata.layers:
    Xraw = layer_to_array("raw", lineage_markers)

    stats_raw = pd.DataFrame({
        "Marker": lineage_markers,
        "Min": Xraw.min(axis=0).round(2),
        "Median": np.median(Xraw, axis=0).round(2),
        "Mean": Xraw.mean(axis=0).round(2),
        "Max": Xraw.max(axis=0).round(2),
        "Std": Xraw.std(axis=0).round(2),
    })

    display_table(
        stats_raw,
        title="Raw intensity statistics",
        subtitle=f"Lineage markers — {adata.n_obs:,} cells"
    )

    save_df_as_pdf(
        stats_raw,
        TABLE_DIR / "raw_intensity_statistics.pdf",
        title=f"{SAMPLE_ID} - raw intensity statistics",
    )

    df_raw = pd.DataFrame(Xraw, columns=lineage_markers)
    df_raw_long = df_raw.melt(var_name="Marker", value_name="Intensity")
    order_raw = df_raw.median().sort_values(ascending=False).index.tolist()

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(
        data=df_raw_long,
        x="Intensity",
        y="Marker",
        order=order_raw,
        orient="h",
        palette="tab10",
        flierprops={"marker": ".", "markersize": 1, "alpha": 0.3},
        ax=ax,
    )
    ax.set_title("Raw Intensity Distribution per Marker", fontsize=13, fontweight="bold")
    ax.set_xlabel("Raw Intensity")
    ax.set_ylabel("")
    plt.tight_layout()
    fig.savefig(DIST_DIR / "raw_intensity_distribution.pdf", dpi=PDF_DPI, bbox_inches="tight")
    if SAVE_PNG_COPY:
        fig.savefig(DIST_DIR / "raw_intensity_distribution.png", dpi=PDF_DPI, bbox_inches="tight")
    plt.show()

else:
    print("Skipping raw intensity statistics: adata.layers['raw'] not found.")


# Arcsinh intensity statistics
Xarc = layer_to_array("arcsinh_full", lineage_markers)

stats_arc = pd.DataFrame({
    "Marker": lineage_markers,
    "Min": Xarc.min(axis=0).round(3),
    "Median": np.median(Xarc, axis=0).round(3),
    "Mean": Xarc.mean(axis=0).round(3),
    "Max": Xarc.max(axis=0).round(3),
    "Std": Xarc.std(axis=0).round(3),
})

display_table(
    stats_arc,
    title="Arcsinh-transformed intensity statistics",
    subtitle=f"Lineage markers — {adata.n_obs:,} cells"
)

save_df_as_pdf(
    stats_arc,
    TABLE_DIR / "arcsinh_intensity_statistics.pdf",
    title=f"{SAMPLE_ID} - arcsinh-transformed intensity statistics",
)

df_arc = pd.DataFrame(Xarc, columns=lineage_markers)
df_arc_long = df_arc.melt(var_name="Marker", value_name="Intensity")
order_arc = df_arc.median().sort_values(ascending=False).index.tolist()

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(
    data=df_arc_long,
    x="Intensity",
    y="Marker",
    order=order_arc,
    orient="h",
    palette="tab10",
    flierprops={"marker": ".", "markersize": 1, "alpha": 0.3},
    ax=ax,
)
ax.set_title("Arcsinh-Transformed Intensity Distribution per Marker", fontsize=13, fontweight="bold")
ax.set_xlabel("Arcsinh Intensity")
ax.set_ylabel("")
plt.tight_layout()
fig.savefig(DIST_DIR / "arcsinh_intensity_distribution.pdf", dpi=PDF_DPI, bbox_inches="tight")
if SAVE_PNG_COPY:
    fig.savefig(DIST_DIR / "arcsinh_intensity_distribution.png", dpi=PDF_DPI, bbox_inches="tight")
plt.show()

In [ ]:
# ==============================================================================
# SECTION 4: UMAP PLOTTING HELPERS
# ==============================================================================

def set_active_umap(selected_umap_key):
    if selected_umap_key not in adata.obsm:
        raise KeyError(
            f"{selected_umap_key} not found in adata.obsm. "
            f"Available keys: {list(adata.obsm.keys())}"
        )
    adata.obsm["X_umap"] = adata.obsm[selected_umap_key].copy()


def expression_layer(norm):
    layer_key = f"{norm}_full"

    if layer_key not in adata.layers:
        raise KeyError(
            f"{layer_key} not found in adata.layers. "
            f"Available layers: {list(adata.layers.keys())}"
        )

    return layer_key


def safe_name(x):
    return str(x).replace(".", "_").replace("/", "_").replace(" ", "_")


def umap_plot_save(color, title, path_pdf, figsize=(9, 8), fontsize=13, **kwargs):
    sc.settings.set_figure_params(figsize=figsize, fontsize=fontsize)

    fig = sc.pl.umap(
        adata,
        color=color if isinstance(color, list) else [color],
        title=title if isinstance(title, list) else [title],
        show=False,
        return_fig=True,
        **kwargs,
    )

    fig.savefig(path_pdf, dpi=PDF_DPI, bbox_inches="tight")

    if SAVE_PNG_COPY:
        fig.savefig(Path(path_pdf).with_suffix(".png"), dpi=PDF_DPI, bbox_inches="tight")

    plt.show()


def marker_umap_grid(norm, md, leiden_res, color_map=MARKER_UMAP_CMAP):
    selected_umap_key = umap_key(norm, md)
    layer_key = expression_layer(norm)
    selected_leiden_key = leiden_key(norm, leiden_res)

    set_active_umap(selected_umap_key)

    if selected_leiden_key not in adata.obs:
        raise KeyError(f"{selected_leiden_key} not found in adata.obs")

    colors = lineage_markers + [selected_leiden_key]
    panel_titles = lineage_markers + [
        f"Leiden clusters\nres={pretty_token(leiden_res)}"
    ]

    fig = sc.pl.umap(
        adata,
        color=colors,
        title=panel_titles,
        ncols=int(MARKER_UMAP_NCOLS),
        size=float(MARKER_UMAP_POINT_SIZE),
        color_map=color_map,
        layer=layer_key,
        show=False,
        return_fig=True,
    )

    fig.suptitle(
        f"Individual marker intensity UMAPs — {norm}-transformed marker intensities | "
        f"expression layer={layer_key} | "
        f"UMAP min_dist={pretty_token(md)} | "
        f"cluster reference: Leiden res={pretty_token(leiden_res)}",
        fontsize=17,
        fontweight="bold",
        y=1.02,
    )

    plt.tight_layout()

    out_pdf = (
        UMAP_MARKER_DIR /
        f"marker_intensity_umap_{norm}_md_{md}_res_{leiden_res}_{safe_name(color_map)}.pdf"
    )

    fig.savefig(out_pdf, dpi=PDF_DPI, bbox_inches="tight")

    if SAVE_PNG_COPY:
        fig.savefig(out_pdf.with_suffix(".png"), dpi=PDF_DPI, bbox_inches="tight")

    plt.show()


def cluster_umap_plot(norm, md, leiden_res):
    selected_umap_key = umap_key(norm, md)
    selected_leiden_key = leiden_key(norm, leiden_res)

    set_active_umap(selected_umap_key)

    if selected_leiden_key not in adata.obs:
        raise KeyError(f"{selected_leiden_key} not found in adata.obs")

    n_clus = n_clusters(selected_leiden_key)

    out_pdf = UMAP_CLUSTER_DIR / f"leiden_umap_{norm}_md_{md}_res_{leiden_res}.pdf"

    umap_plot_save(
        selected_leiden_key,
        (
            f"Leiden clusters from {norm}-transformed scaled lineage markers\n"
            f"UMAP min_dist={pretty_token(md)} | "
            f"Leiden resolution={pretty_token(leiden_res)} | "
            f"{n_clus} clusters"
        ),
        out_pdf,
        figsize=(9, 8),
        fontsize=13,
        size=float(CLUSTER_UMAP_POINT_SIZE),
    )

In [ ]:
# ==============================================================================
# SECTION 4B: MARKER INTENSITY UMAPs
# Individual marker panels + one final Leiden-cluster reference panel
# ==============================================================================
# Only selected parameters are plotted.
# RdYlBu_r is used only; viridis marker UMAPs are intentionally removed.
# ==============================================================================

for _, row in selected_choices.iterrows():
    marker_umap_grid(
        norm=row["normalization"],
        md=row["umap_min_dist_token"],
        leiden_res=row["resolution_token"],
        color_map=MARKER_UMAP_CMAP,
    )

In [ ]:
# ==============================================================================
# SECTION 4C: SELECTED LEIDEN CLUSTER UMAPs
# ==============================================================================
# Two plots by default:
#   - arcsinh selected resolution / selected UMAP min_dist
#   - log1p selected resolution / selected UMAP min_dist
# No other resolutions or UMAP min_dist values are plotted.
# ==============================================================================

for _, row in selected_choices.iterrows():
    cluster_umap_plot(
        norm=row["normalization"],
        md=row["umap_min_dist_token"],
        leiden_res=row["resolution_token"],
    )

**Section 5 note:**  
Dotplots use the same scaled lineage matrix from `.obsm` that RAPIDS used for clustering.  
Matrixplots use `arcsinh_full` because it preserves interpretable marker intensity profiles.  
Only selected Leiden keys are plotted.

In [ ]:
# ==============================================================================
# SECTION 5: MARKER EXPRESSION AND CLUSTER QUALITY CHECKS
# ==============================================================================
# For each selected Leiden key:
#   1. cluster-size barplot
#   2. dotplot from the matching scaled representation used for clustering
#   3. matrixplot from arcsinh_full, with dendrogram
# ==============================================================================


def scaled_rep_from_leiden_key(selected_leiden_key):
    norm_label, _ = parse_leiden_key(selected_leiden_key)
    return f"X_{norm_label}_scaled_lineage"


def dynamic_figsize(selected_leiden_key, marker_width=0.70, cluster_height=0.42, min_width=11, min_height=4.5):
    n_markers = len(lineage_markers)
    n_clus = n_clusters(selected_leiden_key)

    width = max(min_width, n_markers * marker_width)
    height = max(min_height, n_clus * cluster_height)

    return (width, height)


def clean_cluster_title(selected_leiden_key, plot_nature, expression_source):
    norm_label, res_token = parse_leiden_key(selected_leiden_key)
    n_clus = n_clusters(selected_leiden_key)

    return (
        f"{plot_nature}\n"
        f"Clusters from {norm_label}-normalized scaled lineage markers | "
        f"Leiden resolution={pretty_token(res_token)} | "
        f"{n_clus} clusters | "
        f"expression source={expression_source}"
    )


def cluster_size_table(selected_leiden_key):
    counts = adata.obs[selected_leiden_key].astype(str).value_counts()
    counts = counts.loc[sorted(counts.index, key=cluster_sort_key)]

    df = counts.rename_axis("leiden_cluster").reset_index(name="n_cells")
    df["fraction"] = df["n_cells"] / int(adata.n_obs)

    return df


def cluster_size_barplot(selected_leiden_key):
    size_df = cluster_size_table(selected_leiden_key)
    norm_label, res_token = parse_leiden_key(selected_leiden_key)

    fig, ax = plt.subplots(figsize=(max(8, len(size_df) * 0.45), 4.5))

    ax.bar(size_df["leiden_cluster"].astype(str), size_df["n_cells"])

    ax.set_title(
        f"Cluster size check\n"
        f"{norm_label} | Leiden resolution={pretty_token(res_token)} | "
        f"{len(size_df)} clusters",
        fontsize=13,
        pad=12,
    )

    ax.set_xlabel("Leiden cluster")
    ax.set_ylabel("Number of cells / objects")
    ax.tick_params(axis="x", rotation=45)

    plt.tight_layout()

    out_pdf = QC_DIR / f"cluster_size_{norm_label}_res_{res_token}.pdf"
    fig.savefig(out_pdf, dpi=PDF_DPI, bbox_inches="tight")

    if SAVE_PNG_COPY:
        fig.savefig(out_pdf.with_suffix(".png"), dpi=PDF_DPI, bbox_inches="tight")

    plt.show()

    save_df_as_pdf(
        size_df,
        TABLE_DIR / f"cluster_size_table_{norm_label}_res_{res_token}.pdf",
        title=f"{SAMPLE_ID} - cluster sizes: {norm_label}, res={pretty_token(res_token)}",
    )


def make_scaled_dotplot_adata(selected_leiden_key):
    rep_key = scaled_rep_from_leiden_key(selected_leiden_key)

    if rep_key not in adata.obsm:
        raise KeyError(
            f"Missing {rep_key} in adata.obsm. "
            f"Available obsm keys are: {list(adata.obsm.keys())}"
        )

    X_scaled = adata.obsm[rep_key]

    if sparse.issparse(X_scaled):
        X_scaled = X_scaled.toarray()
    else:
        X_scaled = np.asarray(X_scaled)

    if X_scaled.shape[1] != len(lineage_markers):
        raise ValueError(
            f"{rep_key} has {X_scaled.shape[1]} columns, "
            f"but lineage_markers has {len(lineage_markers)} markers."
        )

    adata_dot = ad.AnnData(
        X=X_scaled,
        obs=adata.obs.copy(),
        var=pd.DataFrame(index=lineage_markers),
    )

    return adata_dot, rep_key


def dotplot_for_leiden(selected_leiden_key):
    adata_dot, rep_key = make_scaled_dotplot_adata(selected_leiden_key)
    order = cluster_order(selected_leiden_key)
    norm_label, res_token = parse_leiden_key(selected_leiden_key)

    dot = sc.pl.dotplot(
        adata_dot,
        lineage_markers,
        groupby=selected_leiden_key,
        standard_scale=STANDARD_SCALE,
        categories_order=order,
        cmap=DOTPLOT_CMAP,
        title=clean_cluster_title(
            selected_leiden_key,
            "Marker expression by Leiden cluster — dotplot",
            expression_source=rep_key,
        ),
        figsize=dynamic_figsize(selected_leiden_key),
        show=False,
        return_fig=True,
    )

    out_pdf = QC_DIR / f"dotplot_{norm_label}_res_{res_token}.pdf"
    dot.savefig(out_pdf)

    if SAVE_PNG_COPY:
        dot.savefig(out_pdf.with_suffix(".png"))

    plt.show()


def make_arcsinh_matrixplot_adata(selected_leiden_key):
    if "arcsinh_full" not in adata.layers:
        raise KeyError(
            f"Missing arcsinh_full in adata.layers. "
            f"Available layers are: {list(adata.layers.keys())}"
        )

    X_arc = adata[:, lineage_markers].layers["arcsinh_full"]

    if sparse.issparse(X_arc):
        X_arc = X_arc.toarray()
    else:
        X_arc = np.asarray(X_arc)

    adata_matrix = ad.AnnData(
        X=X_arc,
        obs=adata.obs[[selected_leiden_key]].copy(),
        var=pd.DataFrame(index=lineage_markers),
    )

    adata_matrix.obs[selected_leiden_key] = pd.Categorical(
        adata_matrix.obs[selected_leiden_key].astype(str),
        categories=cluster_order(selected_leiden_key),
    )

    return adata_matrix


def matrixplot_for_leiden(selected_leiden_key):
    norm_label, res_token = parse_leiden_key(selected_leiden_key)
    adata_matrix = make_arcsinh_matrixplot_adata(selected_leiden_key)

    dendro = bool(MATRIXPLOT_DENDROGRAM)

    if dendro:
        sc.tl.dendrogram(adata_matrix, groupby=selected_leiden_key)

    matrix = sc.pl.matrixplot(
        adata_matrix,
        lineage_markers,
        groupby=selected_leiden_key,
        standard_scale=STANDARD_SCALE,
        dendrogram=dendro,
        cmap=MATRIXPLOT_CMAP,
        title=clean_cluster_title(
            selected_leiden_key,
            "Average marker intensity by Leiden cluster — matrixplot",
            expression_source="arcsinh_full",
        ),
        figsize=dynamic_figsize(selected_leiden_key),
        show=False,
        return_fig=True,
    )

    out_pdf = QC_DIR / f"matrixplot_arcsinh_full_{norm_label}_res_{res_token}.pdf"
    matrix.savefig(out_pdf)

    if SAVE_PNG_COPY:
        matrix.savefig(out_pdf.with_suffix(".png"))

    plt.show()


for selected_leiden_key in selected_leiden_keys:
    print("=" * 90)
    print(f"Cluster expression checks: {selected_leiden_key}")
    print("=" * 90)

    cluster_size_barplot(selected_leiden_key)
    dotplot_for_leiden(selected_leiden_key)
    matrixplot_for_leiden(selected_leiden_key)

**Spatial plot note:**  
Spatial scatter plots use all cells in the loaded RAPIDS AnnData and project selected Leiden labels from `adata.obs` onto tissue coordinates. No marker intensity layer is used here.

In [ ]:
# ==============================================================================
# SECTION 6: SPATIAL SCATTER — SELECTED LEIDEN CLUSTERS ON TISSUE COORDINATES
# ==============================================================================
# Only selected Leiden keys are plotted.
# ==============================================================================


def ensure_spatial_coordinates():
    if {"X_centroid", "Y_centroid"}.issubset(adata.obs.columns):
        return

    if "spatial" in adata.obsm:
        coords = adata.obsm["spatial"]

        if isinstance(coords, pd.DataFrame):
            coords = coords.iloc[:, :2].values
        else:
            coords = np.asarray(coords)[:, :2]

        adata.obs["X_centroid"] = coords[:, 0]
        adata.obs["Y_centroid"] = coords[:, 1]
        print("Created X_centroid and Y_centroid from adata.obsm['spatial']")
        return

    raise KeyError(
        "No spatial coordinates found. Expected either "
        "adata.obs['X_centroid'] / adata.obs['Y_centroid'] "
        "or adata.obsm['spatial']."
    )


def spatial_figsize():
    x = adata.obs["X_centroid"].to_numpy()
    y = adata.obs["Y_centroid"].to_numpy()

    x_range = np.nanmax(x) - np.nanmin(x)
    y_range = np.nanmax(y) - np.nanmin(y)

    if y_range == 0:
        return (14, 12)

    aspect = x_range / y_range

    if aspect >= 1:
        width = 15
        height = max(10, width / aspect)
    else:
        height = 14
        width = max(10, height * aspect)

    return (width, height)


def cluster_color_map(selected_leiden_key, palette_name=SPATIAL_PALETTE):
    order = cluster_order(selected_leiden_key)
    cmap = plt.get_cmap(palette_name)

    return {
        cluster: cmap(i % cmap.N)
        for i, cluster in enumerate(order)
    }


def spatial_scatter_leiden(selected_leiden_key):
    if selected_leiden_key not in adata.obs.columns:
        print(f"Skipping {selected_leiden_key}: not found in adata.obs")
        return

    norm_label, res_token = parse_leiden_key(selected_leiden_key)
    order = cluster_order(selected_leiden_key)
    color_map = cluster_color_map(selected_leiden_key)

    fig, ax = plt.subplots(figsize=spatial_figsize())

    cluster_values = adata.obs[selected_leiden_key].astype(str)

    for cluster in order:
        mask = cluster_values == cluster

        ax.scatter(
            adata.obs.loc[mask, "X_centroid"],
            adata.obs.loc[mask, "Y_centroid"],
            s=float(SPATIAL_POINT_SIZE),
            c=[color_map[cluster]],
            alpha=float(SPATIAL_ALPHA),
            linewidths=0,
            label=str(cluster),
            rasterized=True,
        )

    ax.invert_yaxis()
    ax.set_aspect("equal")

    ax.set_xlabel("X centroid", fontsize=12)
    ax.set_ylabel("Y centroid", fontsize=12)

    ax.set_title(
        f"Spatial distribution of Leiden clusters\n"
        f"Clusters from {norm_label}-normalized scaled lineage markers | "
        f"Leiden resolution={pretty_token(res_token)} | "
        f"{n_clusters(selected_leiden_key)} clusters",
        fontsize=15,
        fontweight="bold",
        pad=14,
    )

    legend_cols = max(1, int(np.ceil(n_clusters(selected_leiden_key) / 20)))

    ax.legend(
        title="Leiden cluster",
        bbox_to_anchor=(1.01, 1),
        loc="upper left",
        markerscale=7,
        fontsize=9,
        title_fontsize=10,
        ncol=legend_cols,
        frameon=False,
    )

    plt.tight_layout()

    out_pdf = SPATIAL_DIR / f"spatial_leiden_{norm_label}_res_{res_token}.pdf"
    fig.savefig(out_pdf, dpi=PDF_DPI, bbox_inches="tight")

    if SAVE_PNG_COPY:
        fig.savefig(out_pdf.with_suffix(".png"), dpi=PDF_DPI, bbox_inches="tight")

    plt.show()


ensure_spatial_coordinates()

for selected_leiden_key in selected_leiden_keys:
    print("=" * 90)
    print(f"Spatial cluster check: {selected_leiden_key}")
    print("=" * 90)

    spatial_scatter_leiden(selected_leiden_key)

In [ ]:
# ==============================================================================
# SECTION 7: SESSION SUMMARY
# ==============================================================================
# Nothing is written back into AnnData.
# This notebook is a visualization/reporting step only.
# ==============================================================================

summary = pd.DataFrame([
    {"item": "sample_id", "value": SAMPLE_ID},
    {"item": "sample_index", "value": SAMPLE_INDEX},
    {"item": "input_path", "value": INPUT_PATH},
    {"item": "output_dir", "value": str(OUTPUT_DIR)},
    {"item": "n_cells_loaded", "value": int(adata.n_obs)},
    {"item": "n_markers_total", "value": int(adata.n_vars)},
    {"item": "n_lineage_markers", "value": int(len(lineage_markers))},
    {"item": "selected_normalizations", "value": ", ".join(map(str, SELECTED_NORMALIZATIONS))},
    {"item": "selected_resolutions", "value": ", ".join(pretty_token(x) for x in SELECTED_RESOLUTIONS)},
    {"item": "selected_umap_min_dist", "value": ", ".join(pretty_token(x) for x in SELECTED_UMAP_MIN_DISTS)},
    {"item": "selected_leiden_keys", "value": ", ".join(selected_leiden_keys)},
    {"item": "adata_modified_and_saved", "value": "no"},
])

display_table(summary, title="Cluster exploration session summary")
save_df_as_pdf(
    summary,
    REPORT_DIR / "session_summary.pdf",
    title=f"{SAMPLE_ID} - cluster exploration session summary",
)

print("Cluster exploration completed.")
print("Output folder:")
print(OUTPUT_DIR)